<a href="https://colab.research.google.com/github/SkylarOu9005/AI-Stock-Analysis-Recommendation-Agent/blob/main/Two_Stage_CoT_Stock_Analysis_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 介紹： Chain-of-Thought（CoT）股票分析推理過程
系統整體流程：
</br>
</br>
-------------------[使用者輸入公司名稱或股票代碼]------------------
</br>
</br>[第一階段：資訊蒐集與整理 (Tool)]
</br>- Python程式（用 yfinance）實際去抓今日資料

</br>--------------------[將資料回傳給 LLM]-----------------------
</br>
</br>[第二階段：根據yfinance下載下來的資料分析公司基本面/體質健康程度 (Planning Stage)]
</br>- LLM規劃需要的資料
</br>- 給出公司基本面的資訊
</br>
</br>--------------------[將資料回傳給 LLM]-----------------------
</br>
</br>[第三階段：推理與建議 (Generation Stage)]
</br>- LLM根據資料推理
</br>- 給出建議（建議買入 / 不建議買入）+甚麼人適合買入+ 推理過程
</br>

In [ ]:
import os
from google.colab import userdata

In [ ]:
#【使用 Groq】
api_key = userdata.get('Groq')
os.environ['GROQ_API_KEY']=api_key
provider = "groq"
model = "llama3-70b-8192"

In [ ]:
!pip install aisuite[all]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.9/863.9 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.5/103.5 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.7 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


## 2. 使用 AISuite 的準備

In [ ]:
import aisuite as ai

In [ ]:
provider_writer = "groq"
model_writer = "llama3-70b-8192"

provider_planner = "groq"
model_planner = "llama3-70b-8192"

In [ ]:
def reply(system="請只用台灣習慣的繁體中文回覆。",
      prompt="Hi",
      provider="groq",
      model="llama3-70b-8192"
      ):

    client = ai.Client()

    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt}
    ]


    response = client.chat.completions.create(model=f"{provider}:{model}", messages=messages)

    return response.choices[0].message.content

##  3. 系統架構三階段
</br>第一階段：我在本地使用yfinance抓出近期股價資訊、本益比、殖利率。
</br>第二階段：財務健康初步診斷
</br>第三階段：投資策略導向建議

### 3.1透過yfinance查詢股價資訊
由於LLM本身是「知識截止」的，通常只訓練到2023年初或2023年底。
</br>所以，如果我只是叫它「直接查詢股價」，它是憑記憶回答的，而不是連到最新資料庫去拿。
</br>所以在系統設計上第一階段需要設計一層tool去抓股票資訊（比如今日股價、本益比、殖利率），
</br>並且使用者也要輸入要查詢的日期，系統才會知道現在是幾年幾月幾號。


In [ ]:
import yfinance as yf
from datetime import datetime, timedelta

def get_stock_info_by_date(stock_symbol, date_str):
    stock = yf.Ticker(stock_symbol)

    # 把日期字串轉成 datetime，加一天
    start_date = datetime.strptime(date_str, "%Y-%m-%d")
    end_date = start_date + timedelta(days=1)

    # 抓指定區間的收盤價
    history_data = stock.history(start=start_date.strftime("%Y-%m-%d"), end=end_date.strftime("%Y-%m-%d"))
    if history_data.empty:
        today_price = "暫無資料"
    else:
        today_price = round(history_data['Close'].iloc[0], 2)

    # 取得公司基本資料
    try:
        pe_ratio = stock.info.get('trailingPE', "暫無資料")
        dividend_yield = stock.info.get('dividendYield', "暫無資料")
        if dividend_yield != "暫無資料" and dividend_yield is not None:
            dividend_yield = round(dividend_yield * 100, 2)  # 百分比
    except:
        pe_ratio = "暫無資料"
        dividend_yield = "暫無資料"

    return {
        "today_price": today_price,
        "pe_ratio": pe_ratio,
        "dividend_yield": dividend_yield
    }


### 3.2 設置兩層LLM

In [ ]:
system_planner = """
你是一位專業財務分析師。
請依據使用者提供的公司基本資料，進行財務健康診斷。

診斷請包含以下方面：
1. 本益比是否合理？（過高代表過熱，過低可能被低估）
2. 殖利率是否具有吸引力？（過低代表收益有限）
3. 今日股價是否異常波動？（漲跌幅大於3%需特別注意）

注意事項：
- 僅依據提供資料推理，不要引入外部未知知識。
- 僅做健康評估，不做買賣建議。
- 請用條列式方式回答。
- 只用台灣慣用的繁體中文回答。
"""

In [ ]:
system_writer = """
你是一位專業投資顧問。
請根據使用者提供的公司資料與財務診斷結果，針對不同投資風格，分別給出投資建議。
並告訴投資者股票是否值得買入。

需涵蓋三種投資人：
- 價值型投資者（Value Investor）
- 成長型投資者（Growth Investor）
- 配息型投資者（Dividend Investor）

每一種請說明是否建議買入並簡述理由。
最後總結一句話：「綜合來看，本股票(適合/不適合)買入，適合買入的話最適合哪一類型投資人。」

注意事項：
- 僅依據提供資料推理，不要引入外部未知資訊。
- 回答控制在300字內。
- 請用台灣慣用的繁體中文回答。
"""

#### 主要程式
Step1: 利用get_stock_info_by_date(stock_symbol, query_date)函式查資料。--tool層
</br> Step2: 將資料和system_planner、planning_prompt餵給第一層LLM。
</br> Step3: 將資料和system_writer、generation_prompt餵給第二層LLM。

In [ ]:
def stock_analyzer(stock_symbol, query_date):
    # Step 1: 查資料
    # 第一階段：使用者輸入公司名稱+查詢日期 ➔客觀輸出公司的股價等基本面資訊
    info = get_stock_info_by_date(stock_symbol, query_date)
    stock_info_text = f"""【股票代號】：{stock_symbol}
    【查詢日期】：{query_date}
    【收盤價】：{info['today_price']} 美元
    【本益比】：{info['pe_ratio']}
    【殖利率】：{info['dividend_yield']}%
    """

    # Step 2: 財務健康診斷
    #第二階段：把資料餵給LLM ➔ 讓LLM根據本益比、殖利率、股價變動推理

    planning_prompt = f"""以下是該公司的基本資料：{stock_info_text}請進行財務健康診斷。"""
    stock_diagnosis = reply(system_planner, planning_prompt,
      provider = provider_planner,
      model = model_planner
    )

    # Step 3: 策略型建議
    generation_prompt = f"""以下是該公司的綜合資訊：

    【公司基本資料】：
    {stock_info_text}

    【財務健康診斷】：
    {stock_diagnosis}

    請依照上述資訊，依據不同投資策略，分別給出建議。
    """
    analyze_output = reply(system_writer, generation_prompt,
      provider = provider_writer,
      model = model_writer
    )

    return stock_info_text, stock_diagnosis, analyze_output

### 4. 用 Gradio 打造你的對話機器人 Web App!

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.6/322.6 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 7.7 MB/s eta 0:00:00


In [ ]:
import gradio as gr

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("### 股票分析師 折折🪙 ")
    gr.Markdown("請告訴我一檔你感興趣的股票，我來幫你分析，相信折折，財富高升")
    stock_symbol_input = gr.Textbox(label="請輸入股票代號（例如：TSM、APPL、NVDA、TSLA）")
    query_date_input = gr.Textbox(label="請輸入查詢日期（格式：YYYY-MM-DD，例如：2025-04-26）")

    btn = gr.Button(" 分析開始 ✨")

    with gr.Row():
        out1 = gr.Textbox(label="第一階段：📊 股票基本資料")
    with gr.Row():
        out2 = gr.Textbox(label="第二階段：🧠 財務健康診斷")
    with gr.Row():
        out3 = gr.Textbox(label="第三階段：📣 策略型投資建議")

    btn.click(
      stock_analyzer,
      inputs=[stock_symbol_input, query_date_input],
      outputs=[out1, out2, out3]
    )

In [ ]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abec7bd4f016e25c31.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:yfinance:$NVDA: possibly delisted; no price data found  (1d 2025-04-26 -> 2025-04-27)
